# Medallion Architecture — 01 Bronze to Silver Quality

Goal: understand the next Medallion concept after the basic flow.

In `00_basic`, we learned:

```text
RAW -> BRONZE -> SILVER -> GOLD
              |
              -> QUARANTINE
```

In this notebook we focus only on:

```text
RAW
 |
 v
BRONZE
 |
 v
SILVER QUALITY
 |
 +--> SILVER VALID
 |
 +--> QUARANTINE
```

The main idea:

```text
BRONZE = keep what arrived + add ingestion metadata
SILVER = validate, clean, normalize
QUARANTINE = keep bad rows with reason
```

In [1]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import *

spark = SparkSession.getActiveSession()
if spark is not None:
    spark.stop()

spark = (
    SparkSession.builder
    .appName("medallion-bronze-to-silver-quality")
    .master("spark://spark-master:7077")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.warehouse.dir", "/tmp/spark_medallion_warehouse")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
spark.version

'4.0.2'

## Step 1 — Raw input

We simulate source data.

Some rows are intentionally dirty:

```text
+----------+---------+------------+--------+---------------------+
| event_id | user_id | event_type | amount | event_time          |
+----------+---------+------------+--------+---------------------+
| e1       | u1      | purchase   | 10.0   | 2026-05-02 10:00:00 | valid
| e2       | null    | purchase   | 20.0   | 2026-05-02 10:01:00 | invalid: missing user
| e3       | u2      | purchase   | -5.0   | 2026-05-02 10:02:00 | invalid: negative amount
| e4       | u3      | unknown    | 15.0   | 2026-05-02 10:03:00 | invalid: bad event type
| e5       | u4      | refund     | 5.0    | bad-date            | invalid: bad timestamp
+----------+---------+------------+--------+---------------------+
```

In [2]:
raw_rows = [
    ("e1", "u1", "purchase", 10.0, "2026-05-02 10:00:00"),
    ("e2", None, "purchase", 20.0, "2026-05-02 10:01:00"),
    ("e3", "u2", "purchase", -5.0, "2026-05-02 10:02:00"),
    ("e4", "u3", "unknown", 15.0, "2026-05-02 10:03:00"),
    ("e5", "u4", "refund", 5.0, "bad-date"),
]

raw = spark.createDataFrame(
    raw_rows,
    ["event_id", "user_id", "event_type", "amount", "event_time_raw"]
)

raw.show(truncate=False)

+--------+-------+----------+------+-------------------+
|event_id|user_id|event_type|amount|event_time_raw     |
+--------+-------+----------+------+-------------------+
|e1      |u1     |purchase  |10.0  |2026-05-02 10:00:00|
|e2      |NULL   |purchase  |20.0  |2026-05-02 10:01:00|
|e3      |u2     |purchase  |-5.0  |2026-05-02 10:02:00|
|e4      |u3     |unknown   |15.0  |2026-05-02 10:03:00|
|e5      |u4     |refund    |5.0   |bad-date           |
+--------+-------+----------+------+-------------------+



## Step 2 — Bronze

Bronze should stay close to the source.

But Bronze commonly adds technical metadata:

```text
RAW
 |
 v
BRONZE
- original columns
- source system / file
- ingestion timestamp
- batch id
```

Bronze is not where we enforce business quality.

In [3]:
bronze = (
    raw
    .withColumn("source_system", F.lit("demo-source"))
    .withColumn("source_file", F.lit("events_2026_05_02.json"))
    .withColumn("ingestion_time", F.current_timestamp())
    .withColumn("batch_id", F.lit("batch-001"))
)

bronze.show(truncate=False)
bronze.printSchema()

+--------+-------+----------+------+-------------------+-------------+----------------------+--------------------------+---------+
|event_id|user_id|event_type|amount|event_time_raw     |source_system|source_file           |ingestion_time            |batch_id |
+--------+-------+----------+------+-------------------+-------------+----------------------+--------------------------+---------+
|e1      |u1     |purchase  |10.0  |2026-05-02 10:00:00|demo-source  |events_2026_05_02.json|2026-05-02 13:55:34.336939|batch-001|
|e2      |NULL   |purchase  |20.0  |2026-05-02 10:01:00|demo-source  |events_2026_05_02.json|2026-05-02 13:55:34.336939|batch-001|
|e3      |u2     |purchase  |-5.0  |2026-05-02 10:02:00|demo-source  |events_2026_05_02.json|2026-05-02 13:55:34.336939|batch-001|
|e4      |u3     |unknown   |15.0  |2026-05-02 10:03:00|demo-source  |events_2026_05_02.json|2026-05-02 13:55:34.336939|batch-001|
|e5      |u4     |refund    |5.0   |bad-date           |demo-source  |events_2026_0

## Step 3 — Prepare Silver candidates

Silver is where we start transforming raw strings into useful typed values.

Example:

```text
event_time_raw: string
        |
        v
event_time: timestamp
```

Rows are not filtered yet.  
We first add normalized/parsed columns.

In [4]:
silver_candidates = (
    bronze
    .withColumn("event_time", F.to_timestamp("event_time_raw"))
    .withColumn("event_type", F.lower(F.trim(F.col("event_type"))))
)

silver_candidates.select(
    "event_id",
    "user_id",
    "event_type",
    "amount",
    "event_time_raw",
    "event_time"
).show(truncate=False)

[Stage 4:>                                                          (0 + 1) / 1]

+--------+-------+----------+------+-------------------+-------------------+
|event_id|user_id|event_type|amount|event_time_raw     |event_time         |
+--------+-------+----------+------+-------------------+-------------------+
|e1      |u1     |purchase  |10.0  |2026-05-02 10:00:00|2026-05-02 10:00:00|
|e2      |NULL   |purchase  |20.0  |2026-05-02 10:01:00|2026-05-02 10:01:00|
|e3      |u2     |purchase  |-5.0  |2026-05-02 10:02:00|2026-05-02 10:02:00|
|e4      |u3     |unknown   |15.0  |2026-05-02 10:03:00|2026-05-02 10:03:00|
|e5      |u4     |refund    |5.0   |bad-date           |NULL               |
+--------+-------+----------+------+-------------------+-------------------+



## Step 4 — Quality rules

We define simple quality rules.

```text
Rule 1: event_id is required
Rule 2: user_id is required
Rule 3: event_type must be purchase or refund
Rule 4: amount must be >= 0
Rule 5: event_time must be parsed successfully
```

Instead of immediately filtering, we add `error_reason`.

In [5]:
allowed_event_types = ["purchase", "refund"]

quality_checked = (
    silver_candidates
    .withColumn(
        "error_reason",
        F.when(F.col("event_id").isNull(), F.lit("missing_event_id"))
         .when(F.col("user_id").isNull(), F.lit("missing_user_id"))
         .when(~F.col("event_type").isin(allowed_event_types), F.lit("invalid_event_type"))
         .when(F.col("amount") < 0, F.lit("negative_amount"))
         .when(F.col("event_time").isNull(), F.lit("invalid_event_time"))
    )
    .withColumn("is_valid", F.col("error_reason").isNull())
)

quality_checked.select(
    "event_id",
    "user_id",
    "event_type",
    "amount",
    "event_time_raw",
    "event_time",
    "is_valid",
    "error_reason"
).show(truncate=False)

+--------+-------+----------+------+-------------------+-------------------+--------+------------------+
|event_id|user_id|event_type|amount|event_time_raw     |event_time         |is_valid|error_reason      |
+--------+-------+----------+------+-------------------+-------------------+--------+------------------+
|e1      |u1     |purchase  |10.0  |2026-05-02 10:00:00|2026-05-02 10:00:00|true    |NULL              |
|e2      |NULL   |purchase  |20.0  |2026-05-02 10:01:00|2026-05-02 10:01:00|false   |missing_user_id   |
|e3      |u2     |purchase  |-5.0  |2026-05-02 10:02:00|2026-05-02 10:02:00|false   |negative_amount   |
|e4      |u3     |unknown   |15.0  |2026-05-02 10:03:00|2026-05-02 10:03:00|false   |invalid_event_type|
|e5      |u4     |refund    |5.0   |bad-date           |NULL               |false   |invalid_event_time|
+--------+-------+----------+------+-------------------+-------------------+--------+------------------+



## Step 5 — Silver valid rows

Silver valid rows are clean enough for downstream processing.

```text
QUALITY CHECKED
      |
      v
SILVER VALID
```

We keep useful metadata, but we remove temporary raw helper columns if needed.

In [6]:
silver_valid = (
    quality_checked
    .filter(F.col("is_valid"))
    .select(
        "event_id",
        "user_id",
        "event_type",
        "amount",
        "event_time",
        "source_system",
        "source_file",
        "ingestion_time",
        "batch_id"
    )
)

silver_valid.show(truncate=False)

+--------+-------+----------+------+-------------------+-------------+----------------------+--------------------------+---------+
|event_id|user_id|event_type|amount|event_time         |source_system|source_file           |ingestion_time            |batch_id |
+--------+-------+----------+------+-------------------+-------------+----------------------+--------------------------+---------+
|e1      |u1     |purchase  |10.0  |2026-05-02 10:00:00|demo-source  |events_2026_05_02.json|2026-05-02 13:55:37.221975|batch-001|
+--------+-------+----------+------+-------------------+-------------+----------------------+--------------------------+---------+



## Step 6 — Quarantine rows

Invalid rows should not disappear.

They go to quarantine with a clear reason.

```text
QUALITY CHECKED
      |
      v
QUARANTINE
```

This is useful for debugging, monitoring, and reprocessing.

In [7]:
quarantine = (
    quality_checked
    .filter(~F.col("is_valid"))
    .select(
        "event_id",
        "user_id",
        "event_type",
        "amount",
        "event_time_raw",
        "source_system",
        "source_file",
        "ingestion_time",
        "batch_id",
        "error_reason"
    )
)

quarantine.show(truncate=False)

+--------+-------+----------+------+-------------------+-------------+----------------------+--------------------------+---------+------------------+
|event_id|user_id|event_type|amount|event_time_raw     |source_system|source_file           |ingestion_time            |batch_id |error_reason      |
+--------+-------+----------+------+-------------------+-------------+----------------------+--------------------------+---------+------------------+
|e2      |NULL   |purchase  |20.0  |2026-05-02 10:01:00|demo-source  |events_2026_05_02.json|2026-05-02 13:55:37.646042|batch-001|missing_user_id   |
|e3      |u2     |purchase  |-5.0  |2026-05-02 10:02:00|demo-source  |events_2026_05_02.json|2026-05-02 13:55:37.646042|batch-001|negative_amount   |
|e4      |u3     |unknown   |15.0  |2026-05-02 10:03:00|demo-source  |events_2026_05_02.json|2026-05-02 13:55:37.646042|batch-001|invalid_event_type|
|e5      |u4     |refund    |5.0   |bad-date           |demo-source  |events_2026_05_02.json|2026-05

## Step 7 — Summary counts

A useful pipeline habit:

```text
How many rows arrived?
How many became Silver?
How many went to Quarantine?
```

In [8]:
summary = spark.createDataFrame(
    [
        ("bronze_rows", bronze.count()),
        ("silver_valid_rows", silver_valid.count()),
        ("quarantine_rows", quarantine.count()),
    ],
    ["metric", "value"]
)

summary.show()

+-----------------+-----+
|           metric|value|
+-----------------+-----+
|      bronze_rows|    5|
|silver_valid_rows|    1|
|  quarantine_rows|    4|
+-----------------+-----+



## Final mental model

```text
RAW
 |
 v
BRONZE
- keep source shape
- add ingestion metadata
- do not enforce heavy business rules

BRONZE
 |
 v
SILVER CANDIDATES
- parse timestamps
- normalize text
- prepare typed columns

SILVER QUALITY
 |
 +--> SILVER VALID
 |    - clean rows
 |    - valid event types
 |    - parsed timestamps
 |
 +--> QUARANTINE
      - invalid rows
      - error_reason
      - source metadata
```

This is the practical Bronze → Silver quality pattern.

In [9]:
spark.stop()